## Introduction notebook to retrieve current odds from admiral and input optimal tip into kicktipp

In [1]:
import os
from random import sample, seed

In [2]:
# if packages don't yet exist, download here
# !pip install selenium, beautifulsoup4, pandas

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
main_path   = "https://sports.admiral.at/de/sportwetten/fussball/deutschland/bundesliga/"

In [5]:
### parameters
start_match = 2
end_match = 2
matchday = 34
### parameters

In [6]:
def abbr_to_team(abbr, return_mapping = False):
    mapping = {
        "BVB": "Dortmund",
        "BMG": "Gladbach",
        "FCB": "Bayern München",
        "RBL": "RB Leipzig",
        "SCF": "Freiburg",
        "SGE": "Eintracht Frankfurt",
        "VFB": "VfB Stuttgart",
        "VFL": "VfL Wolfsburg",
        "TSG": "Hoffenheim",
        "FCU": "Union Berlin",
        "FCA": "Augsburg",
        "SVW": "Werder",
        "F95": "FSV Mainz 05",
        "KOE": "1. FC Köln",
        "LEV": "Leverkusen",
        "STP": "St. Pauli",
        "HEI'": "1. FC Heidenheim",
        "HSV": "HSV"
    }
    if return_mapping: 
        assert abbr == None
        return mapping
    return mapping.get(abbr, abbr)  # Return the abbreviation itself if not found


def kicktipp_to_abr(name):
    mapping = {
        "Borussia Dortmund": "BVB",
        "Bor. Mönchengladbach": "BMG",
        "FC Bayern München": "FCB",
        "RB Leipzig": "RBL",
        "SC Freiburg": "SCF",
        "Eintracht Frankfurt": "SGE",
        "VfB Stuttgart": "VFB",
        "VfL Wolfsburg": "VFL",
        "1899 Hoffenheim": "TSG",
        "1. FC Union Berlin": "FCU",
        "FC Augsburg": "FCA",
        "Werder Bremen": "SVW",
        "FSV Mainz 05": "F95",
        "1. FC Köln": "KOE",
        "Bayer 04 Leverkusen": "LEV",
        "FC St. Pauli": "STP",
        "1. FC Heidenheim 1846": "HEI",
        "Hamburger SV": "HSV"
    }
    return mapping.get(name, name)  # Return the name itself if not found

In [7]:
def compute_exp(proj_results,probabilities,final_result):
  score = 0
  hs_final, as_final = final_result.split(":")
  hs_final, as_final = int(hs_final), int(as_final)
  for idx, res in enumerate(proj_results[:-1]):
    hs_res, as_res = res.split(":")
    hs_res, as_res = int(hs_res), int(as_res)
    if res == final_result:
      score = score + probabilities[idx] * 4
    elif (hs_res - as_res) == (hs_final - as_final):
      if (hs_res - as_res) != 0:
        score = score + probabilities[idx] * 3
      else:
        score = score + probabilities[idx] * 2
    elif ((hs_res - as_res > 0) & (hs_final - as_final > 0)):
      score = score + probabilities[idx] * 2
    elif ((hs_res - as_res < 0) & (hs_final - as_final < 0)):
      score = score + probabilities[idx] * 2
    else:
      pass
  return score


def odds_to_probs(odds):
  probsum = 0
  p = []
  p_clean = []
  for odd in odds:
    prob = 1/odd
    probsum = probsum + prob
    p.append(prob)
  for prob in p:
    final = prob/probsum
    p_clean.append(final)
  return p_clean


def return_maximum(proj_results,odds):
  maxima = []
  maximum = 0
  maxima_idx = 0
  probabilities = odds_to_probs(odds)
  for idx, result in enumerate(proj_results):
    exp = compute_exp(proj_results,probabilities,result)
    if exp > maximum:
      maxima = []
      maximum = exp
      maxima.append(idx)
    elif exp == maximum:
      maxima.append(idx)
  if len(maxima) > 1:
      max_results = [proj_results[idx] for idx in maxima]
      print(f"There exist multiple identical maxima, in total: {len(maxima)}")
      print(f"These are the results: {max_results}")
      seed(27079)
      maxima_idx = sample(range(len(maxima)),1)[0]
  return proj_results[maxima[maxima_idx]]

In [8]:
def credentials():
    with open("login.txt", "r") as login_file:
        for line in login_file:
            input, credential = line.replace(" ","").replace("\n","").split(":")
            if input == "USERNAME": username = credential
            elif input == "PASSWORD": password = credential
            elif input == "TIPROUND": tipround = credential
            else: raise NotImplementedError
    return username, password, tipround

In [9]:
def scrape_odds(matchday, home_team, away_team):
    
    home_teams = [home_team]
    away_teams = [away_team]

    for ht, at in zip(home_teams, away_teams):
        driver = webdriver.Chrome()

        home_team = ht.replace(" ", "-").replace(".", "")
        away_team = at.replace(" ", "-").replace(".", "")
        complete_path = main_path + home_team + "-vs-" + away_team + "?tab=filter_1"
        complete_path = complete_path.replace("ß", "ss").replace("ä", "ae").replace("ö", "oe").replace("ü", "ue")
        complete_path = complete_path.replace("Ä", "Ae").replace("Ö", "Oe").replace("Ü", "Ue")

        complete_path = complete_path.lower()

        driver.get(complete_path)

        try:
            accept_cookies_button = WebDriverWait(driver, 20).until(
                EC.any_of(
                    EC.element_to_be_clickable((By.XPATH, "//button[text()='Nicht erforderliche ablehnen']")),
                    EC.element_to_be_clickable((By.XPATH, "//button[text()='Alle akzeptieren und schließen']")),
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Nicht erforderliche ablehnen')]")),
                    EC.element_to_be_clickable((By.XPATH, "//*[contains(@class, 'accept')]"))
                )
            )
            accept_cookies_button.click()
        except Exception as e:
            print("No cookie button found or error clicking it:", e)

        try:
            resultat_div = WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.XPATH, "//div[starts-with(@id, 'event_market-board_market_sr:market:41/sr:match:')]"))
            )
        except Exception as e:
            print("Resultat div not found:", e)
            driver.quit()
            return

        soup = BeautifulSoup(resultat_div.get_attribute('outerHTML'), 'html.parser')

        rows = []
        for sel in soup.find_all("asw-marketboard-selection"):
            label_tag = sel.find("span", id=lambda v: v and v.endswith("_selection_name"))
            odd_tag   = sel.find("span", id=lambda v: v and v.endswith("_selection_odds"))
            if label_tag and odd_tag:
                label = label_tag.get_text(strip=True)
                odd = odd_tag.get_text(strip=True).replace(",", ".")   # convert 14,00 → 14.00
                rows.append({"Result": label, "Odd": float(odd)})

        df = pd.DataFrame(rows)

        home_name = home_team.replace("-", "_").lower().replace("ü", "ue").replace("ä", "ae").replace("ö", "oe")
        away_name = away_team.replace("-", "_").lower().replace("ü", "ue").replace("ä", "ae").replace("ö", "oe")
        if not os.path.exists(f"matchday_{matchday}"):
            os.mkdir(f"matchday_{matchday}")
        df.to_csv(f"matchday_{matchday}/{home_name}_vs_{away_name}.csv", index=False)

        odds = df['Odd'].tolist()
        proj_results = df['Result'].tolist()


        decimal_odds = list(map(float, odds))

        print(f"Matchday {matchday}: {ht} vs {at}")
        print(return_maximum(proj_results,decimal_odds))

    driver.quit()

In [10]:
driver = webdriver.Chrome()

def get_teams(match_number):
    row_xpath = f'//*[@id="tippabgabeSpiele"]/tbody/tr[{match_number}]'

    driver.set_window_size(1080, 1920)
    driver.execute_script("document.body.style.zoom='80%'")    

    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.XPATH, row_xpath + '/td[2]')))
    
    home_team = driver.find_element(By.XPATH, row_xpath + '/td[2]').text
    away_team = driver.find_element(By.XPATH, row_xpath + '/td[3]').text

    return home_team, away_team


def login():
    USERNAME, PASSWORD, TIPROUND = credentials()

    tipround  = f"https://www.kicktipp.de/{TIPROUND}/tippabgabe"
    driver.get(tipround)

    wait = WebDriverWait(driver, 10)

    username_input = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="kennung"]')))
    username_input.clear()
    username_input.send_keys(USERNAME)

    password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="passwort"]')))
    password_input.clear()
    password_input.send_keys(PASSWORD)

    login_button = driver.find_element(By.XPATH, '//*[@id="loginFormular"]/div/div[3]/div/button')
    login_button.click()

    print("Login button clicked. Waiting for navigation link...")

    try:
        tippabgabe_link = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="navigation"]/div[5]/a')))
        tippabgabe_link.click()
        print("Navigated to Tippabgabe successfully.")
    except Exception as e:
        print(f"Error navigating to Tippabgabe after login: {e}")


def get_matches(start_match = 1, end_match = 9):
    login()
    matches = []
    for match in range(start_match, end_match + 1):
        home, away = get_teams(match)
        ht, at = kicktipp_to_abr(home), kicktipp_to_abr(away)
        matches.append((ht,at))
    return matches


matches = get_matches(start_match=start_match, end_match=end_match)

for (ht, at) in matches:
    home, away = abbr_to_team(ht), abbr_to_team(at)
    scrape_odds(matchday, home, away)

driver.quit()

Login button clicked. Waiting for navigation link...
Navigated to Tippabgabe successfully.
Matchday 34: Leverkusen vs HSV
2:0


## Input the tips into kicktipp

In [11]:
def retrieve_relevant_dataframe(matchday, home, away):
    dir = f"matchday_{matchday}/"
    
    home_team = abbr_to_team(kicktipp_to_abr(home)).replace(" ", "-").replace(".", "")
    away_team = abbr_to_team(kicktipp_to_abr(away)).replace(" ", "-").replace(".", "")

    home_name = home_team.replace("-", "_").lower().replace("ü", "ue").replace("ä", "ae").replace("ö", "oe")
    away_name = away_team.replace("-", "_").lower().replace("ü", "ue").replace("ä", "ae").replace("ö", "oe")

    file = f"{home_name}_vs_{away_name}.csv"

    df = pd.read_csv(os.path.join(dir, file))

    odds = df['Odd'].tolist()
    proj_results = df['Result'].tolist()

    decimal_odds = list(map(float, odds))
    tipp = str(return_maximum(proj_results, decimal_odds))

    home_tipp, away_tipp = tipp.split(":")

    return home_tipp, away_tipp


def tipp(matchday, start_match = 1, end_match = 9):
    for match_number in range(start_match, end_match + 1):
        try:
            home, away = get_teams(match_number=match_number)

            row_xpath = f'//*[@id="tippabgabeSpiele"]/tbody/tr[{match_number}]'

            row = driver.find_element(By.XPATH, row_xpath)
            home_input = row.find_element(By.XPATH, ".//input[contains(@id, 'heimTipp')]")
            away_input = row.find_element(By.XPATH, ".//input[contains(@id, 'gastTipp')]")

        
            home_tipp, away_tipp = retrieve_relevant_dataframe(matchday, home, away)

            print(f"Tip: {home} vs {away}")
            print(home_tipp, ":", away_tipp)

            home_input.clear()
            home_input.send_keys(str(home_tipp))
            away_input.clear()
            away_input.send_keys(str(away_tipp))

        except Exception as e:
            print(f"Error processing row {match_number}: {e}")


    try:
        submit_button = driver.find_element(By.XPATH, '//*[@id="tippabgabeForm"]/div/div/div/button')
        submit_button.click()
    except Exception as e:
        print(f"Error clicking submit button: {e}")

In [13]:
driver = webdriver.Chrome()
login()
tipp(matchday = matchday, start_match=start_match, end_match = end_match)
driver.quit()

Login button clicked. Waiting for navigation link...
Navigated to Tippabgabe successfully.
Tip: Bayer 04 Leverkusen vs Hamburger SV
2 : 0
